In [1]:
import pandas as pd
from gflownet.proxy.mol_utils import random_split
from rdkit import Chem
from utils import obtain_run, get_config, scale_rew, load_proxy_sol
from gflownet.utils.sqlite_log import read_all_results

In [3]:
FRAGMENTS: list[tuple[str, list[int]]] = [
    ("Br", [0]),
    ("C", [0]),
    ("C#N", [0]),
    ("C1=CCCCC1", [0, 2, 3]),
    ("C1=CNC=CC1", [0, 2]),
    ("C1CC1", [0]),
    ("C1CCCC1", [0]),
    ("C1CCCCC1", [0, 1, 2, 3, 4, 5]),
    ("C1CCNC1", [0, 2, 3, 4]),
    ("C1CCNCC1", [0, 1, 3]),
    ("C1CCOC1", [0, 1, 2, 4]),
    ("C1CCOCC1", [0, 1, 2, 4, 5]),
    ("C1CNCCN1", [2, 5]),
    ("C1COCCN1", [5]),
    ("C1COCC[NH2+]1", [5]),
    ("C=C", [0, 1]),
    ("C=C(C)C", [0]),
    ("C=CC", [0, 1]),
    ("C=N", [0]),
    ("C=O", [0]),
    ("CC", [0, 1]),
    ("CC(C)C", [1]),
    ("CC(C)O", [1]),
    ("CC(N)=O", [2]),
    ("CC=O", [1]),
    ("CCC", [1]),
    ("CCO", [1]),
    ("CN", [0, 1]),
    ("CNC", [1]),
    ("CNC(C)=O", [0]),
    ("CNC=O", [0, 2]),
    ("CO", [0, 1]),
    ("CS", [0]),
    ("C[NH3+]", [0]),
    ("C[SH2+]", [1]),
    ("Cl", [0]),
    ("F", [0]),
    ("FC(F)F", [1]),
    ("I", [0]),
    ("N", [0]),
    ("N=CN", [1]),
    ("NC=O", [0, 1]),
    ("N[SH](=O)=O", [1]),
    ("O", [0]),
    ("O=CNO", [1]),
    ("O=CO", [1]),
    ("O=C[O-]", [1]),
    ("O=PO", [1]),
    ("O=P[O-]", [1]),
    ("O=S=O", [1]),
    ("O=[NH+][O-]", [1]),
    ("O=[PH](O)O", [1]),
    ("O=[PH]([O-])O", [1]),
    ("O=[SH](=O)O", [1]),
    ("O=[SH](=O)[O-]", [1]),
    ("O=c1[nH]cnc2[nH]cnc12", [3, 6]),
    ("O=c1[nH]cnc2c1NCCN2", [8, 3]),
    ("O=c1cc[nH]c(=O)[nH]1", [2, 4]),
    ("O=c1nc2[nH]c3ccccc3nc-2c(=O)[nH]1", [8, 4, 7]),
    ("O=c1nccc[nH]1", [3, 6]),
    ("S", [0]),
    ("c1cc[nH+]cc1", [1, 3]),
    ("c1cc[nH]c1", [0, 2]),
    ("c1ccc2[nH]ccc2c1", [6]),
    ("c1ccc2ccccc2c1", [0, 2]),
    ("c1ccccc1", [0, 1, 2, 3, 4, 5]),
    ("c1ccncc1", [0, 1, 2, 4, 5]),
    ("c1ccsc1", [2, 4]),
    ("c1cn[nH]c1", [0, 1, 3, 4]),
    ("c1cncnc1", [0, 1, 3, 5]),
    ("c1cscn1", [0, 3]),
    ("c1ncc2nc[nH]c2n1", [2, 6]),
]

data_url = r"https://raw.githubusercontent.com/CesareWang/Predictors-for-15-Environmental-Endpoints/main/predictors/data/SW.csv"
smiles_data = pd.read_csv(data_url, index_col=0)["smiles"]
train_id, _ , _ = random_split(0.8, smiles_data)
train_smiles = smiles_data[train_id]
frag_list = []
cutoff = round(0.01 * len(train_smiles))
for fragment in FRAGMENTS:
    frag = fragment[0]
    exist = 0
    for smiles in train_smiles:
        fr = Chem.MolFromSmiles(frag)
        larger_mol = Chem.MolFromSmiles(smiles)
        if larger_mol.HasSubstructMatch(fr):
            exist+=1
    frag_list.append((frag, exist))
    if frag_list[-1][1] < cutoff:
        print(f"Fragment {frag} has less than {cutoff} existences in the training set.")

Fragment C1=CNC=CC1 has less than 42 existences in the training set.
Fragment C1CCNC1 has less than 42 existences in the training set.
Fragment C1CNCCN1 has less than 42 existences in the training set.
Fragment C1COCCN1 has less than 42 existences in the training set.
Fragment C1COCC[NH2+]1 has less than 42 existences in the training set.
Fragment C[SH2+] has less than 42 existences in the training set.
Fragment O=CNO has less than 42 existences in the training set.
Fragment O=C[O-] has less than 42 existences in the training set.
Fragment O=P[O-] has less than 42 existences in the training set.
Fragment O=[PH]([O-])O has less than 42 existences in the training set.
Fragment O=[SH](=O)[O-] has less than 42 existences in the training set.
Fragment O=c1[nH]cnc2[nH]cnc12 has less than 42 existences in the training set.
Fragment O=c1[nH]cnc2c1NCCN2 has less than 42 existences in the training set.
Fragment O=c1cc[nH]c(=O)[nH]1 has less than 42 existences in the training set.
Fragment O=c1nc

In [ ]:
total_atoms = sum(Chem.MolFromSmiles(smiles).GetNumAtoms() for smiles in train_smiles if Chem.MolFromSmiles(smiles))
average_atoms = total_atoms / len(smiles_data)
print(f"Average number of atoms per SMILES in the dataset: {average_atoms}")

Average number of atoms per SMILES: 11.656226271829917


In [21]:
import pandas as pd
data_url = r"https://raw.githubusercontent.com/CesareWang/Predictors-for-15-Environmental-Endpoints/main/predictors/data/SW.csv"
smiles_data = pd.read_csv(data_url, index_col=0)["smiles"]
sol_data = list(pd.read_csv(data_url, index_col=0)["active"])
ids = ["fm_experiment(lr=1e-4)"]
for id in ids:
    run_path = obtain_run(id)
    # config = get_config(id)
    results = read_all_results(run_path / "final")

In [23]:
total_atoms = sum(Chem.MolFromSmiles(smiles).GetNumAtoms() for smiles in results["smi"] if Chem.MolFromSmiles(smiles))
average_atoms = total_atoms / len(smiles_data)
print(f"Average number of atoms per SMILES in the dataset: {average_atoms}")

Average number of atoms per SMILES in the dataset: 35.674069855732725


In [22]:
from rdkit import Chem
from rdkit.Chem import MACCSkeys
from rdkit import DataStructs


def generate_maccs_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fingerprint = MACCSkeys.GenMACCSKeys(mol)
    return fingerprint


smiles = results["smi"]
maccs_fingerprints = [generate_maccs_fingerprint(smile) for smile in smiles]

import numpy as np

# Compute pairwise Tanimoto similarities
n_samples = len(maccs_fingerprints)
similarity_matrix = np.zeros((n_samples, n_samples))

for i in range(n_samples):
    for j in range(i, n_samples):
        similarity = DataStructs.TanimotoSimilarity(maccs_fingerprints[i], maccs_fingerprints[j])
        similarity_matrix[i, j] = similarity
        similarity_matrix[j, i] = similarity  # Symmetric matrix

# Calculate average pairwise similarity
# Extract upper triangular part of the similarity matrix (excluding diagonal)
upper_triangular = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]
# Calculate average and standard deviation
average_similarity = np.mean(upper_triangular)

# Calculate the average similarity for each molecule
average_similarities = similarity_matrix.mean(axis=1)

sorted_indices = np.argsort(average_similarities)

# Retrieve the indices of the 100 most dissimilar molecules
most_dissimilar_indices = sorted_indices[:100]

# Retrieve the SMILES strings of the 100 most dissimilar molecules
most_dissimilar_smiles_list = [smiles[idx] for idx in most_dissimilar_indices]

print("The 100 most dissimilar molecules are:")
for idx, smi in enumerate(most_dissimilar_smiles_list, start=1):
    print(f"{idx}: {smi}")

The 100 most dissimilar molecules are:
1: ClCl
2: N#CI
3: N#CBr
4: O=CBr
5: CC(C)Br
6: CC#N
7: FCS
8: FC(F)(F)CS
9: CCS
10: CC(=O)F
11: CC(C)=CC(C)C
12: N#CC=N
13: CC(C)=CF
14: CN
15: O=P(O)(O)C(F)(F)F
16: CC(O)F
17: CC(C)(C)P(=O)(O)O
18: CC(C)=CC=O
19: CC(C)=C[SH](=O)=O
20: NC[NH3+]
21: O=C(NI)[PH](=O)O
22: CC(=O)C=C(C)C=N
23: N=CC(=O)NC(F)(F)F
24: [NH3+]C[PH](=O)O
25: CC(O)N(C)C
26: O=[N+]([O-])C1CC1
27: CC(=O)NCC(C)(C)C
28: CC(=CCl)C1OC(I)C(C(C)(C)O)C1S(N)(=O)=O
29: C1CCCCC1
30: N#CCOCC[N+](=O)[O-]
31: CC(C)c1ccc(-c2cccc(-c3ccc(-c4cccc(S)c4)cc3)c2)cc1
32: CC(=CC(=O)NCC1CC1)[N+](=O)[O-]
33: O=C(O)c1cc(Br)cc2ccccc12
34: CC(N)=Cc1cc(C(=O)O)cc(C(C)(C)C)c1
35: CC(=O)Nc1cc(C)ccc1[N+](=O)[O-]
36: NS(=O)(=O)c1c(Br)cc(Br)c(C(=O)O)c1C1CC1
37: N#Cc1ccc(I)cc1-c1ccc([PH](=O)O)c(S(N)(=O)=O)c1
38: CC(C)(O)c1cc(C=CC2CC2)c(S(=O)(=O)O)cc1C[NH3+]
39: NS(=O)(=O)c1ccccc1-c1ccc(-c2ccc(-c3ccccc3)cc2Cl)cc1
40: CC(C)=Cc1cccc(-c2cccc(-c3cccc(-c4ccc(-c5ccccc5)cc4)c3)c2)c1
41: O=C(COC=CF)Nc1cccc(F)c1
42: CCC=C

In [46]:
average_similarity

0.18421738862979994

In [11]:
list(smiles_data)

['C=O',
 'CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O)C(=O)CO',
 'CC(=O)OCC(=O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3C(O)CC21C',
 'CC(=O)OCC(=O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3C(=O)CC21C',
 'CCC1(c2ccccc2)C(=O)NC(=O)NC1=O',
 'CCC1(CC)C(=O)NC(=O)N(C)C1=O',
 'O=P1(N(CCCl)CCCl)NCCCO1',
 'CC(O)C(=O)O',
 'CC12CCC(=O)C=C1CCC1C2C(O)CC2(C)C(C(=O)CO)CCC12',
 'CC12CCC(=O)C=C1CCC1C2C(O)CC2(C)C1CCC2(O)C(=O)CO',
 'CC12C=CC(=O)C=C1CCC1C2C(O)CC2(C)C1CCC2(O)C(=O)CO',
 'CC12CCC3c4ccc(O)cc4CCC3C1CCC2O',
 'Clc1ccc(C(c2ccc(Cl)cc2)C(Cl)(Cl)Cl)cc1',
 'O=C(O)c1c(Cl)cccc1Cl',
 'O=C(O)c1c(Cl)ccc(Cl)c1Cl',
 'c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34',
 'CCCCC1C(=O)N(c2ccccc2)N(c2ccccc2)C1=O',
 'O=C1CCC(N2C(=O)c3ccccc3C2=O)C(=O)N1',
 'COC(=O)C1C(OC(=O)c2ccccc2)CC2CCC1N2C',
 'S=c1[nH]cnc2[nH]cnc12',
 'CNCCCN1c2ccccc2CCc2ccccc21',
 'CN(C)CCC=C1c2ccccc2CCc2ccccc21',
 'CN(C)CCCN1c2ccccc2CCc2ccccc21',
 'CN(C)CCCN1c2ccccc2Sc2ccc(Cl)cc21',
 'COC(=O)C1C2CC3c4[nH]c5cc(OC)ccc5c4CCN3CC2CC(OC(=O)c2cc(OC)c(OC)c(OC)c2)C1OC',
 'NCCc1c[nH]c2

In [15]:
sorted_results = results.sort_values(by="r", ascending=False)


In [18]:
from rdkit.Chem import Descriptors

# Function to calculate SAS score
def calculate_sas_score(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return Descriptors.qed(mol)  # Using QED as a proxy for SAS score
    return None

sorted_results = results["smi"]
# Calculate SAS scores for sorted_results
sas_scores = sorted_results.apply(calculate_sas_score)

# Combine SMILES and SAS scores into a DataFrame
sas_df = pd.DataFrame({'SMILES': sorted_results, 'SAS_Score': sas_scores})

# Sort by SAS score from lowest to highest
sorted_sas_df = sas_df.sort_values(by='SAS_Score', ascending=True)
sorted_sas_df.reset_index(drop=True, inplace=True)
sorted_sas_df

,SMILES,SAS_Score
0,IC1C=CCCC1c1ccccc1-c1cc(-c2cccc(-c3ccccc3)c2)c...,0.116187
1,O=[N+]([O-])c1cc(-c2ccc(-c3cc(-c4ccc5ccccc5c4)...,0.119023
2,IC1CCC(c2cc(-c3cc(-c4cc(-c5ccccc5)c5ccccc5c4)c...,0.129776
3,O=[N+]([O-])c1cc(-c2cccc(-c3ccc(I)cc3)c2)cc(-c...,0.133967
4,CC(=O)c1ccc(-c2cc(I)nc(-c3ccc(-c4ccc5ccccc5c4)...,0.135078
...,...,...
6395,CC(C)(C)c1cc(-c2cccc(CN)c2)c(C2CC2)c([SH](=O)=...,0.826263
6396,CC(O)c1cc([PH](=O)O)ccc1-c1ccc(C#N)c(C(C)(C)C)c1,0.833353
6397,C=Cc1ccc(C#N)c(S)c1-c1ccccc1CO,0.833659
6398,CC(O)=CNC(=O)c1ccc(-c2ccncc2F)cc1,0.844103


In [14]:
results["smi"]

0       CCc1cc(-c2cc(-c3ccc4ccccc4c3)cc(-c3ccccc3C3=CC...
1       CC=Cc1ccc(-c2cc(CC)ccc2-c2ccc3ccccc3c2)cc1-c1c...
2       CC=Cc1cccc(-c2cnccc2-c2cc(-c3ccc4ccccc4c3)cc(-...
3       c1ccc(-c2ncccc2-c2cc(C3CCCCC3)ccn2)c(NCC2CCOC2)c1
4       C(=Cc1cc(-c2ccccc2)ccc1-c1cccc(-c2ccccc2)c1)c1...
                              ...                        
6395    O=Cc1cc(-c2cccc3ccccc23)ccc1-c1cccc(-c2cccc(-c...
6396    c1ccc(-c2ccc3ccccc3c2)c(-c2ccc(-c3ccc(-c4cccc5...
6397     CCc1ccccc1-c1ccccc1-c1ccccc1-c1cccc(-c2ccccc2)c1
6398    C1=CC(c2ccc(-c3cccc(-c4cc(-c5cccc6ccccc56)cc(-...
6399    Cc1ccccc1-c1cc(C2CCCC(c3cccc(-c4ccc5ccccc5c4)n...
Name: smi, Length: 6400, dtype: object